In [ ]:
import nltk

from nltk.tag.brill import *                            # импортируем классы и функции для работы с Brill tagger
import nltk.tag.brill_trainer as bt                     # модуль для обучения Brill tagger
from nltk.corpus import brown                           # импортируем корпус brown, который содержит размеченные тексты на английском
from sklearn.model_selection import train_test_split    # импортируем функцию для разделения данных на тренировочкую и тестовую выборки
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
nltk.download('brown')
nltk.download('punkt')

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
Template._cleartemplates()                                                                    # очистка ранее загруженных или установленных шаблонов для Брилла
templates = fntbl37()                                                                         # создаем набор шаблонов для Брилла, конкретно этот - стандартный набор из 37 правил для создания теггера
tagged_sentences = brown.tagged_sents(categories = ['news', 'learned', 'adventure'])          # загружаем размеченные предложения из корпуса brown (только тексты категорий news, learned, adventure)
train_data, test_data = train_test_split(tagged_sentences, test_size=0.2, random_state=42)    # разделяем данные на тренировочную и тестовую выборки с фиксированным случайным состоянием

In [ ]:
initial_tagger = nltk.tag.BigramTagger(train_data)                                          # создаем началью теггер на основе биграммной модели - он используется как базовый для дальнейшего обучения Брилла
trainer = bt.BrillTaggerTrainer(initial_tagger, templates, trace=3)                         # инициализация тренера
brill_tagger_en = trainer.train(train_data, max_rules=1000)                                 # обучение Брилла с ипользованием тренировочной выборки

TBL train (fast) (seqs: 13595; tokens: 282035; tpls: 37; min score: 2; min acc: None)
Finding initial useful rules...
    Found 837440 useful rules.

           B      |
   S   F   r   O  |        Score = Fixed - Broken
   c   i   o   t  |  R     Fixed = num tags changed incorrect -> correct
   o   x   k   h  |  u     Broken = num tags changed correct -> incorrect
   r   e   e   e  |  l     Other = num tags changed incorrect -> incorrect
   e   d   n   r  |  e
------------------+-------------------------------------------------------
70487048   037340  | None->NN if Pos:None@[1]
32713271   0  64  | None->. if Pos:NN@[-3,-2,-1]
28802880   0  15  | NN->AT if Word:the@[0]
21082108   0   5  | NN->, if Word:,@[0]
15631563   0  35  | NN->IN if Word:of@[0]
11981198   0  11  | NN->CC if Word:and@[0]
10821203 121   4  | TO->IN if Pos:NN@[1,2]
 924 924   0  24  | NN->IN if Word:in@[0]
 903 903   0   2  | NN->AT if Word:a@[0] & Pos:NN@[1]
 836 866  30 922  | NN->IN if Pos:NN@[-1] & Pos:AT@[1]
 59

In [ ]:
print('Точность на тестовом наборе для английского: ', brill_tagger_en.evaluate(test_data)) # оценка точности

<ipython-input-46-102f19419fc8>:1: DeprecationWarning: 
  Function evaluate() has been deprecated.  Use accuracy(gold)
  instead.
  print('Точность на тестовом наборе для английского: ', brill_tagger_en.evaluate(test_data)) # оценка точности


Точность на тестовом наборе для английского:  0.7884844227157378


# Для русского

In [ ]:
!wget -q https://raw.githubusercontent.com/appling2024/data/refs/heads/main/GSD_train.txt
!wget -q https://raw.githubusercontent.com/appling2024/data/refs/heads/main/GSD_test.txt
# загружаем два текстовых файла из указанного репозитория: первый - тренировочный набор предложения для русского языка, второй - тестовый

train_sents_ru = []
with open('GSD_train.txt', 'r', encoding='utf-8') as f:
  current_sentence = []
  for line in f:
    line = line.strip().split('\t')
    if len(line) > 1:
      current_sentence.append((line[1], line[3]))
    elif current_sentence:
      train_sents_ru.append(current_sentence)
      current_sentence = []
# train_sents_ru становится списком размеченных предложений

In [ ]:
Template._cleartemplates() # опять очищаем шаблон
templates = nltk.tag.brill.nltkdemo18() # загружаем стандартный набор из 18 правил для Брилла
initial_tagger_ru = nltk.tag.UnigramTagger(train_sents_ru) # создаем начальный униграммный теггер, который даст наиболее вероятный тег для каждого слова на основе данных тренировочного корпуса
trainer_ru = bt.BrillTaggerTrainer(initial_tagger_ru, templates, trace=3) # инициализация тренера
brill_tagger_ru = trainer_ru.train(train_sents_ru, max_rules=850) # тренировка
# в результате мы создаем обученный Брилл, который уточняет разметку и улучшает начальную униграммную модель

TBL train (fast) (seqs: 4974; tokens: 96949; tpls: 18; min score: 2; min acc: None)
Finding initial useful rules...
    Found 21151 useful rules.

           B      |
   S   F   r   O  |        Score = Fixed - Broken
   c   i   o   t  |  R     Fixed = num tags changed incorrect -> correct
   o   x   k   h  |  u     Broken = num tags changed correct -> incorrect
   r   e   e   e  |  l     Other = num tags changed incorrect -> incorrect
   e   d   n   r  |  e
------------------+-------------------------------------------------------
  80 104  24   0  | NUM->ADJ if Word:года@[1,2,3]
  53  53   0   0  | DET->PRON if Pos:SCONJ@[2]
  48  61  13   3  | PART->CCONJ if Word:а@[-1]
  27  43  16   1  | DET->PRON if Pos:VERB@[1]
  21  27   6   0  | DET->PRON if Pos:VERB@[-1] & Pos:ADP@[1]
  19  22   3   1  | ADP->ADV if Word:так@[-1]
  19  27   8   0  | ADV->SCONJ if Word:как@[1]
  15  15   0   0  | NUM->ADJ if Word:сентября@[1]
  12  21   9   0  | PRON->DET if Word:же@[1]
  11  16   5   4  | DET-

# Тестирование на своих фразах

In [ ]:
print(brill_tagger_ru.tag(nltk.word_tokenize("Я куда-то шел, очень хотел спать, все время спотыкался, хотел пить и опять хотел спать. Сейчас я тоже хочу спать")))

[('Я', 'PRON'), ('куда-то', None), ('шел', None), (',', 'PUNCT'), ('очень', 'ADV'), ('хотел', 'VERB'), ('спать', 'VERB'), (',', 'PUNCT'), ('все', 'DET'), ('время', 'NOUN'), ('спотыкался', None), (',', 'PUNCT'), ('хотел', 'VERB'), ('пить', 'VERB'), ('и', 'CCONJ'), ('опять', 'ADV'), ('хотел', 'VERB'), ('спать', 'VERB'), ('.', 'PUNCT'), ('Сейчас', 'ADV'), ('я', 'PRON'), ('тоже', 'PART'), ('хочу', None), ('спать', 'VERB')]


In [ ]:
# PRON  --- местоимение
# PUNCT --- знак препинания
# ADV   --- наречие
# VERB  --- глагол
# DET   --- детерминатив
# NOUN  --- существительное
# CCONJ --- сочинительный союз
# PART  --- частица

# Тестирование на тестовом GSD

In [ ]:
correct, incorrect = 0, 0
errors = []
# создаем счетчик правильно и ошибочно размеченных токенов; errors - список, который будет содержать пары для всех ошибок теггера

with open('GSD_test.txt', 'r', encoding='utf-8') as f:
  current_sentence = []
  gold_sentence = []
  for line in f:
    line = line.strip().split('\t')
    if len(line) > 1:
      current_sentence.append(line[1])
      gold_sentence.append((line[1], line[3]))
    elif current_sentence:
      predicted_tags = brill_tagger_ru.tag(current_sentence)
      for gold, predicted in zip(gold_sentence, predicted_tags):
        if gold[1] == predicted[1]:
          correct += 1
        else:
          incorrect += 1
          errors.append((gold, predicted))
        current_sentence = []
        gold_sentence = []

# если теги в gold и predicted совпадают, то счетчик correct увеличивается. Если не совпадают, то увеличивается счетчик incorrect, а ошибка добавляется в список errors

In [ ]:
# подводим итоги
all_tokens = correct + incorrect
accuracy = correct / all_tokens if all_tokens > 0 else 0
print('Общее число токенов: ', all_tokens)
print('Доля правильно размеченных токенов от общего числа: ', accuracy)

Общее число токенов:  1012
Доля правильно размеченных токенов от общего числа:  0.7322134387351779


In [ ]:
# выводим ошибки
print('\nОшибки (первые 10):')
for error in errors[:10]:
  print(f'Ожидалось: {error[0]}, Предсказано: {[1]}')


Ошибки (первые 10):
Ожидалось: ('резервный', 'ADJ'), Предсказано: [1]
Ожидалось: ('Черка', 'PROPN'), Предсказано: [1]
Ожидалось: ('6.00', 'NUM'), Предсказано: [1]
Ожидалось: ('00.20', 'NUM'), Предсказано: [1]
Ожидалось: ('Стал', 'VERB'), Предсказано: [1]
Ожидалось: ('секретариата', 'NOUN'), Предсказано: [1]
Ожидалось: ('SSP', 'X'), Предсказано: [1]
Ожидалось: ('Secretaría', 'X'), Предсказано: [1]
Ожидалось: ('Seguridad', 'X'), Предсказано: [1]
Ожидалось: ('Pública', 'X'), Предсказано: [1]
